# Accessing and Downloading GloFAS Data from the Copernicus Emergency Management Service (CEMS)\n

In this tutorial you will learn how to access and download GloFAS river discharge data from the CEMS Early Warning Data Store (EWDS). We use a clear, step-by-step workflow aimed at beginners.\n\nWe will use **earthkit-data** to inspect and open downloaded files because it reduces boilerplate code, supports many data formats, and integrates well with `xarray`, `pandas`, and `NumPy`.\n

## Learning Objectives\n\nBy the end of this notebook you will be able to:\n\n- Explain what EWDS is and how it relates to the Copernicus Climate Data Store APIs\n- Authenticate and prepare a GloFAS data request\n- Download GloFAS reanalysis data from EWDS\n- Open and inspect the downloaded data with `earthkit.data`\n- Convert data to `xarray` and create a simple discharge time-series plot\n

## 1. Background: EWDS and GloFAS\n\n- **EWDS** is the access point for Copernicus emergency management early warning datasets.\n- **GloFAS** provides global hydrological information, including historical reanalysis and forecast products.\n- Data requests are submitted through an API using your CDS/EWDS account credentials.\n

## 2. Import libraries\n\nWe import the core libraries used in this training notebook.\n

In [ ]:
# Data access and processing libraries\nimport cdsapi\nimport earthkit.data as ekd\nimport xarray as xr\nimport pandas as pd\nimport numpy as np\n\n# Plotting library\nimport matplotlib.pyplot as plt\n

## 3. Authentication\n\nBefore downloading data, make sure your API credentials are configured.\n\nYou can store credentials in your `~/.cdsapirc` file, or provide them directly in the cell below.\n

In [ ]:
# Leave these as None if your ~/.cdsapirc is already configured\ncdsapi_url = None\ncdsapi_key = None\n\n# Create an API client\nclient = cdsapi.Client(url=cdsapi_url, key=cdsapi_key)\n

## 4. Define a GloFAS reanalysis request\n\nHere we request daily river discharge (`river_discharge_in_the_last_24_hours`) for a small region and limited period. This keeps the file small for training.\n

In [ ]:
# Dataset name in the EWDS catalogue\ndataset_name = "cems-glofas-historical"\n\n# Output file name\noutput_filename = "glofas_reanalysis_sample.grib"\n\n# Request parameters\nrequest = {\n    "system_version": ["version_4_0"],\n    "hydrological_model": ["lisflood"],\n    "product_type": ["consolidated"],\n    "variable": ["river_discharge_in_the_last_24_hours"],\n    "hyear": ["2020"],\n    "hmonth": ["01", "02"],\n    "hday": ["01", "02", "03", "04", "05"],\n    "area": [52, 5, 48, 10],  # North, West, South, East\n    "data_format": "grib",\n    "download_format": "unarchived",\n}\n

## 5. Download data from EWDS\n\nRun the next cell to submit the request and save data locally.\n

In [ ]:
# Submit the request and download the file\nclient.retrieve(dataset_name, request).download(output_filename)\n\nprint(f"Download complete: {output_filename}")\n

## 6. Open and inspect data with earthkit-data\n\nThis is where `earthkit.data` is especially helpful. It opens many formats using one interface.\n

In [ ]:
# Load the downloaded GRIB file with earthkit-data\nek_data = ekd.from_source("file", output_filename)\n\n# Inspect the object\nek_data\n

Now we convert to an `xarray.Dataset` because it is convenient for time selection and plotting.\n

In [ ]:
# Convert earthkit object to xarray\nds = ek_data.to_xarray()\n\n# Display dataset summary\nds\n

## 7. Extract a point time series\n\nWe now extract the nearest grid cell to a chosen location and create a dataframe for inspection.\n

In [ ]:
# Set a target location (example: central Europe)\ntarget_latitude = 50.0\ntarget_longitude = 7.0\n\n# Identify the discharge variable name used in GloFAS files\ndischarge_variable_name = "dis24"\n\n# Select nearest grid point\npoint_series = ds[discharge_variable_name].sel(\n    latitude=target_latitude,\n    longitude=target_longitude,\n    method="nearest",\n)\n\n# Convert to a pandas DataFrame\npoint_dataframe = point_series.to_dataframe().reset_index()\n\n# Show first rows\npoint_dataframe.head()\n

## 8. Plot a hydrograph\n\nA hydrograph helps us inspect discharge variation over time.\n

In [ ]:
# Create a line plot of discharge through time\nfig, ax = plt.subplots(figsize=(10, 5))\nax.plot(point_dataframe["valid_time"], point_dataframe[discharge_variable_name], label="Discharge")\nax.set_title("GloFAS Reanalysis Daily Discharge at Selected Grid Point")\nax.set_xlabel("Date")\nax.set_ylabel("River discharge (m³/s)")\nax.grid(True)\nax.legend()\nplt.show()\n

## 9. Interpret the result\n\nIn this small sample, each point on the hydrograph represents daily simulated river discharge from the GloFAS reanalysis. In operational flood workflows, this is often compared with thresholds (for example Q90) to identify potential flood signal conditions.\n

## Exercises\n\n### Exercise 1\nDownload data for a different year and compare the hydrograph shape.\n\n**Hints:**\n- Update `hyear` in the request dictionary.\n- Keep the same location first, then try a new location.\n\n### Exercise 2\nCalculate monthly mean discharge for the selected point.\n\n**Hints:**\n- Convert `valid_time` to datetime if needed.\n- Group by month and calculate mean values.\n- Plot monthly means with labelled axes.\n\n### Exercise 3\nEstimate a simple high-flow threshold using the 90th percentile (Q90) for your sample and count exceedances.\n\n**Hints:**\n- Use `np.percentile(..., 90)` on the discharge series.\n- Create a boolean column for exceedances.\n- Count `True` values and interpret the result for flood awareness.\n

## Take-home messages\n\n- EWDS provides access to GloFAS datasets through API requests.\n- A clear request definition helps you control file size and workflow speed.\n- `earthkit.data` is a practical first step for loading and inspecting hydrological files.\n- Converting to `xarray` and `pandas` supports analysis and visualisation for flood forecasting workflows.\n